### Importing pandas and loading the df

In [1]:
import pandas as pd
df= pd.read_csv("sample.csv")

In [2]:
# Loading NRC lexicon
nrc_df = pd.read_csv(
    "https://raw.githubusercontent.com/dinbav/LeXmo/master/NRC-Emotion-Lexicon-Wordlevel-v0.92.txt",
    sep='\t', header=None, names=['word', 'emotion', 'association']
)
nrc_df = nrc_df[nrc_df['association'] == 1]  # only positive associations
nrc_dict = nrc_df.groupby('word')['emotion'].apply(list).to_dict()

print(f"NRC lexicon loaded: {len(nrc_dict)} words")



NRC lexicon loaded: 6468 words


In [3]:
# Scoring function
def get_emotion_scores(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return {}
    words = text.lower().split()
    counts = {}
    for word in words:
        if word in nrc_dict:
            for emotion in nrc_dict[word]:
                counts[emotion] = counts.get(emotion, 0) + 1
    total = len(words)
    return {k: v / total for k, v in counts.items()}

In [4]:

# Applying the function to the df
df["emotion_scores"] = df["article_text"].apply(get_emotion_scores)

# ── 4. Expand dict into clean columns ────────────────────────────────────
emotion_df = df["emotion_scores"].apply(pd.Series).fillna(0)
df = pd.concat([df, emotion_df], axis=1)
df.drop(columns=["emotion_scores"], inplace=True)


In [5]:
df.head(20)

,id,primary_country,country_freq,article_text,anger,fear,negative,sadness,surprise,anticipation,positive,trust,disgust,joy
0,3deecee02dba9cb11c3dda429f4f3181696915f1,US,"{'US': 4, 'NZ': 2, 'WS': 3, 'AS': 1, 'TO': 1}",(CNN) -- When an earthquake threatens to turn ...,0.013850,0.035088,0.023084,0.009234,0.011080,0.017544,0.048938,0.025854,0.004617,0.005540
1,c327a45bce74b6e46e2b8e26c7c358dc5d211133,ID,"{'ID': 12, 'NO': 4}",(CNN) -- One of the world's largest pulp and p...,0.013208,0.013208,0.022642,0.007547,0.003774,0.015094,0.033962,0.013208,0.001887,0.003774
2,0c583c71f1ceddb9569f6d8588f62108ca645b91,KP,"{'KP': 4, 'CN': 3}","(CNN) -- Kim Jong Un, successor to his father'...",0.008876,0.019231,0.026627,0.013314,0.013314,0.025148,0.038462,0.028107,0.008876,0.005917
3,f7ef47b6d6d9963c4801bb0c472629238a38740b,IQ,{'IQ': 12},Baghdad (CNN) -- At least 16 people were kille...,0.032941,0.049412,0.056471,0.030588,0.014118,0.014118,0.044706,0.035294,0.018824,0.009412
4,e439517292eaeab7ffead2af6ba826017b2d2f61,CU,{'CU': 2},Atlanta (CNN) -- Former U.S. President Jimmy C...,0.008475,0.004237,0.012712,0.008475,0.004237,0.029661,0.076271,0.038136,0.004237,0.008475
5,4d0dc17c4ad398906ac103f476d9355f169a7d87,US,"{'US': 5, 'AU': 1, 'BS': 1}",(CNN) -- New York's famous skyline may be gett...,0.000000,0.003650,0.000000,0.000000,0.003650,0.021898,0.058394,0.007299,0.000000,0.007299
6,cf9840f55e863768468593d034a8d2e8b60f2a3c,GB,"{'IE': 4, 'GB': 6, 'IT': 3, 'FR': 2}",Ireland kept alive their Six Nations champions...,0.003344,0.003344,0.020067,0.010033,0.010033,0.020067,0.030100,0.020067,0.000000,0.020067
7,3a00cfd7ef02b9e00ca88fc42ce732fae239c419,US,{'US': 1},(CNN) -- A Massachusetts grand jury has indict...,0.022305,0.029740,0.018587,0.014870,0.011152,0.011152,0.037175,0.040892,0.026022,0.014870
8,a89edbfe9c26f5f423d9448eda6e1e9bcb4dabf0,AF,"{'AF': 1, 'GB': 1, 'DE': 1}","(CNN) -- ""Call of Duty: Modern Warfare 3"" cont...",0.021875,0.025000,0.037500,0.014063,0.007812,0.020313,0.043750,0.032813,0.012500,0.017188
9,0722eb694b993eb3731c1237ac89ce09aa46948c,US,{'US': 1},(CNN) -- Time is running out for congressional...,0.012270,0.034765,0.040900,0.028630,0.010225,0.028630,0.049080,0.040900,0.010225,0.020450


In [6]:
df.describe()

,anger,fear,negative,sadness,surprise,anticipation,positive,trust,disgust,joy
count,250.000000,250.000000,250.000000,250.000000,250.000000,250.000000,250.000000,250.000000,250.000000,250.000000
mean,0.014507,0.022108,0.028590,0.013513,0.008678,0.018701,0.045686,0.031333,0.007788,0.013520
std,0.011195,0.015604,0.014903,0.008641,0.005481,0.008354,0.014359,0.013387,0.006192,0.008134
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.015152,0.000000,0.000000,0.000000
25%,0.006050,0.009875,0.017328,0.007411,0.004830,0.012909,0.036053,0.022326,0.003054,0.007851
50%,0.012289,0.017674,0.026730,0.011888,0.008002,0.017773,0.043644,0.029606,0.006553,0.012768
75%,0.020026,0.031270,0.037675,0.018368,0.011380,0.023388,0.053836,0.038246,0.011171,0.017917
max,0.054018,0.082677,0.074766,0.040959,0.026906,0.053512,0.096154,0.073684,0.040959,0.058496


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               250 non-null    object 
 1   primary_country  226 non-null    object 
 2   country_freq     250 non-null    object 
 3   article_text     250 non-null    object 
 4   anger            250 non-null    float64
 5   fear             250 non-null    float64
 6   negative         250 non-null    float64
 7   sadness          250 non-null    float64
 8   surprise         250 non-null    float64
 9   anticipation     250 non-null    float64
 10  positive         250 non-null    float64
 11  trust            250 non-null    float64
 12  disgust          250 non-null    float64
 13  joy              250 non-null    float64
dtypes: float64(10), object(4)
memory usage: 27.5+ KB


In [9]:
df.head(1)

,id,primary_country,country_freq,article_text,anger,fear,negative,sadness,surprise,anticipation,positive,trust,disgust,joy
0,3deecee02dba9cb11c3dda429f4f3181696915f1,US,"{'US': 4, 'NZ': 2, 'WS': 3, 'AS': 1, 'TO': 1}",(CNN) -- When an earthquake threatens to turn ...,0.01385,0.035088,0.023084,0.009234,0.01108,0.017544,0.048938,0.025854,0.004617,0.00554


### Labelling check

In [10]:
# Check highest fear article
most_fear_idx = df['fear'].idxmax()
print(f"Fear score: {df.loc[most_fear_idx, 'fear']:.4f}")
print(f"Country: {df.loc[most_fear_idx, 'primary_country']}")
print()
print(df.loc[most_fear_idx, 'article_text'][:500])  # first 500 chars

Fear score: 0.0827
Country: US

New Orleans, Louisiana (CNN) -- Four New Orleans police officers accused of killing two men after Hurricane Katrina are scheduled to appear in federal courtrooms Friday. The officers are charged with multiple counts of conspiracy, weapons and civil rights violations in connection with the well-publicized 2005 shootings in the infamous Danziger Bridge incident. The hearings Friday will determine whether the officers will be held in jail until the trial, according to court documents. Three of the 


In [11]:
for emotion in ['anger', 'joy', 'trust', 'sadness']:
    idx = df[emotion].idxmax()
    print(f"\n── Highest {emotion.upper()} (score: {df.loc[idx, emotion]:.4f}) ──")
    print(df.loc[idx, 'article_text'][:300])
    print()


── Highest ANGER (score: 0.0540) ──
(CNN) -- When most Americans think about heroic efforts that save lives and keep communities safe from gun violence, I suspect they picture someone with a badge, gun or bullet-proof vest who, with similarly equipped colleagues, busts down doors in pursuit of criminal thugs. I salute the fine officer


── Highest JOY (score: 0.0585) ──
(CNN) -- Somali gunmen on Wednesday released a British aid worker kidnapped last week while working for Save the Children. The aid worker, named as Zimbabwe-born Frans Barnard, was freed after tribal elders negotiated with his captors, Save the Children confirmed. Barnard was safe and well and had b


── Highest TRUST (score: 0.0737) ──
(CNN) -- Administrative Professionals Week is coming to a close, and though the term "secretary" is fraught with negative meaning for some, there have been stellar examples of efficiency, smarts and loyalty shown by administrative assistants both in real life and on the silver screen th

In [12]:
def explain_emotion_scores(text, nrc_dict):
    words = text.lower().split()
    matches = {}
    for word in words:
        if word in nrc_dict:
            for emotion in nrc_dict[word]:
                if emotion not in matches:
                    matches[emotion] = []
                matches[emotion].append(word)
    return matches

# Inspect a specific article
idx = 0  # change this to any row
matches = explain_emotion_scores(df.loc[idx, 'article_text'], nrc_dict)

for emotion, words in matches.items():
    print(f"{emotion:15s}: {words[:10]}")  # show up to 10 trigger words


anger          : ['earthquake', 'fits', 'frenetic', 'disaster', 'earthquake', 'earthquake', 'delay', 'caution', 'interrupt', 'strike']
fear           : ['earthquake', 'warning', 'warning', 'warning', 'warning', 'evacuate', 'frenetic', 'warning', 'disaster', 'warning']
negative       : ['earthquake', 'wait', 'fits', 'evacuate', 'tragic', 'frenetic', 'disaster', 'earthquake', 'earthquake', 'delay']
sadness        : ['earthquake', 'disaster', 'earthquake', 'earthquake', 'delay', 'disaster', 'earthquake', 'case', 'feeling', 'feeling']
surprise       : ['earthquake', 'frenetic', 'disaster', 'earthquake', 'earthquake', 'interrupt', 'lucky', 'disaster', 'earthquake', 'feeling']
anticipation   : ['wait', 'thought', 'time', 'time', 'time', 'result', 'caution', 'time', 'time', 'scientist']
positive       : ['pacific', 'center', 'job', 'director', 'coast', 'coast', 'pacific', 'center', 'pacific', 'sea']
trust          : ['center', 'director', 'center', 'director', 'center', 'real', 'center', 'ope

### Moral Foundations Theory lexicon

In [20]:
import re

def parse_wmodel(filepath):
    """
    Parse the .wmodel MFD file format:
    [CATEGORIZATION]
    CARE.VIRTUE          <- category header (no tab)
        WORD (1)         <- word entry (has tab)
    """
    word_to_foundation = {}
    current_category = None
    in_categorization = False

    # Known MFD categories to look for
    mfd_categories = [
        'CARE.VIRTUE', 'CARE.VICE',
        'FAIRNESS.VIRTUE', 'FAIRNESS.VICE',
        'LOYALTY.VIRTUE', 'LOYALTY.VICE',
        'AUTHORITY.VIRTUE', 'AUTHORITY.VICE',
        'SANCTITY.VIRTUE', 'SANCTITY.VICE'
    ]

    with open(filepath, 'r', encoding='utf-8-sig', errors='ignore') as f:
        for line in f:
            line = line.rstrip('\r\n')

            # Enter categorization section
            if line.strip() == '[CATEGORIZATION]':
                in_categorization = True
                continue

            # Exit categorization section
            if in_categorization and line.startswith('[') and line.strip() != '[CATEGORIZATION]':
                in_categorization = False
                continue

            if not in_categorization:
                continue

            # Category header — no leading tab, matches known MFD category
            if not line.startswith('\t') and line.strip() in mfd_categories:
                current_category = line.strip().lower()  # e.g. 'care.virtue'
                continue

            # Word entry — has leading tab
            if line.startswith('\t') and current_category:
                # Format: "\tWORD (1)"  ->  extract just the word
                word = line.strip().split('(')[0].strip().lower()
                if word:
                    word_to_foundation[word] = current_category

    return word_to_foundation

# Load it
# Mac example
word_to_foundation = parse_wmodel("Moral Foundations Dictionary.wmodel 2")

print(f"Loaded: {len(word_to_foundation)} words")
print(f"\nCategories: {set(word_to_foundation.values())}")
print(f"\nSample words per category:")
import pandas as pd
sample_df = pd.DataFrame(list(word_to_foundation.items()), columns=['word', 'foundation'])
for cat, grp in sample_df.groupby('foundation'):
    print(f"  {cat:25s}: {list(grp['word'])[:5]}")

Loaded: 2041 words

Categories: {'loyalty.vice', 'care.virtue', 'sanctity.virtue', 'authority.vice', 'fairness.virtue', 'loyalty.virtue', 'sanctity.vice', 'fairness.vice', 'care.vice', 'authority.virtue'}

Sample words per category:
  authority.vice           : ['rebel', 'rebellion', 'rebellions', 'rebels', 'treacherous']
  authority.virtue         : ['protect', 'protected', 'protecting', 'protection', 'protector']
  care.vice                : ['vulnerability', 'vulnerable', 'wound', 'wounded', 'wounding']
  care.virtue              : ['alleviate', 'alleviated', 'alleviates', 'alleviating', 'alleviation']
  fairness.vice            : ['exploit', 'exploitation', 'exploited', 'exploiter', 'exploiters']
  fairness.virtue          : ['avenge', 'avenged', 'avenger', 'avengers', 'avenges']
  loyalty.vice             : ['betray', 'betrayed', 'betrayer', 'betrayers', 'betraying']
  loyalty.virtue           : ['all_for_one', 'allied', 'allies', 'ally', 'belong']
  sanctity.vice            : ['a

In [21]:
df.head(20)

,id,primary_country,country_freq,article_text,anger,fear,negative,sadness,surprise,anticipation,positive,trust,disgust,joy
0,3deecee02dba9cb11c3dda429f4f3181696915f1,US,"{'US': 4, 'NZ': 2, 'WS': 3, 'AS': 1, 'TO': 1}",(CNN) -- When an earthquake threatens to turn ...,0.013850,0.035088,0.023084,0.009234,0.011080,0.017544,0.048938,0.025854,0.004617,0.005540
1,c327a45bce74b6e46e2b8e26c7c358dc5d211133,ID,"{'ID': 12, 'NO': 4}",(CNN) -- One of the world's largest pulp and p...,0.013208,0.013208,0.022642,0.007547,0.003774,0.015094,0.033962,0.013208,0.001887,0.003774
2,0c583c71f1ceddb9569f6d8588f62108ca645b91,KP,"{'KP': 4, 'CN': 3}","(CNN) -- Kim Jong Un, successor to his father'...",0.008876,0.019231,0.026627,0.013314,0.013314,0.025148,0.038462,0.028107,0.008876,0.005917
3,f7ef47b6d6d9963c4801bb0c472629238a38740b,IQ,{'IQ': 12},Baghdad (CNN) -- At least 16 people were kille...,0.032941,0.049412,0.056471,0.030588,0.014118,0.014118,0.044706,0.035294,0.018824,0.009412
4,e439517292eaeab7ffead2af6ba826017b2d2f61,CU,{'CU': 2},Atlanta (CNN) -- Former U.S. President Jimmy C...,0.008475,0.004237,0.012712,0.008475,0.004237,0.029661,0.076271,0.038136,0.004237,0.008475
5,4d0dc17c4ad398906ac103f476d9355f169a7d87,US,"{'US': 5, 'AU': 1, 'BS': 1}",(CNN) -- New York's famous skyline may be gett...,0.000000,0.003650,0.000000,0.000000,0.003650,0.021898,0.058394,0.007299,0.000000,0.007299
6,cf9840f55e863768468593d034a8d2e8b60f2a3c,GB,"{'IE': 4, 'GB': 6, 'IT': 3, 'FR': 2}",Ireland kept alive their Six Nations champions...,0.003344,0.003344,0.020067,0.010033,0.010033,0.020067,0.030100,0.020067,0.000000,0.020067
7,3a00cfd7ef02b9e00ca88fc42ce732fae239c419,US,{'US': 1},(CNN) -- A Massachusetts grand jury has indict...,0.022305,0.029740,0.018587,0.014870,0.011152,0.011152,0.037175,0.040892,0.026022,0.014870
8,a89edbfe9c26f5f423d9448eda6e1e9bcb4dabf0,AF,"{'AF': 1, 'GB': 1, 'DE': 1}","(CNN) -- ""Call of Duty: Modern Warfare 3"" cont...",0.021875,0.025000,0.037500,0.014063,0.007812,0.020313,0.043750,0.032813,0.012500,0.017188
9,0722eb694b993eb3731c1237ac89ce09aa46948c,US,{'US': 1},(CNN) -- Time is running out for congressional...,0.012270,0.034765,0.040900,0.028630,0.010225,0.028630,0.049080,0.040900,0.010225,0.020450
